# 07e — Train & Evaluate (Fixed-Composition CV, Pooled Bogor+Warsaw)

Fifth parallel branch alongside `07` (spatial k-fold), `07b` (bootstrap),
`07c` (60/20/20 random repeats), and `07d` (70:30, no val). Uses this
project's new pooled-cities track (`04b_vocab_unification.ipynb` +
`05b_dataset_assembly_pooled.ipynb`) as its data source instead of a
single city's `dataset_index.parquet`.

**What's fixed vs. what moves, exactly:**
- **Fixed across every repeat:** the split *scheme* — 65/15/20
  train/val/test, stratified JOINTLY by (label x city) via
  `train._stratified_split_by_city`, so every repeat's train/val/test
  match the source data's class balance AND each city's share of the
  dataset, by construction.
- **Fixed across every scenario AT a given repeat:** scenario A-G
  (including the new F, see below) all draw the SAME train/val/test rows
  at a given `repeat` index — guaranteed by `run_scenario_cv_repeats`
  using the identical `seed + repeat` for every scenario's call, same
  contract `run_scenario_random_repeats` already has for A-E/G in `07c`.
  This makes scenario comparisons at a given repeat directly comparable
  (literally the same held-out points), not just comparable in aggregate.
- **Moves each repeat:** which specific rows land in train vs. val vs.
  test — a fresh stratified draw per `repeat_seed = seed + repeat`, same
  as `07c`. This is NOT k-fold (no guarantee every point appears in test
  exactly once across the 5 repeats) and NOT bootstrap (no resampling
  with replacement — every point used exactly once per repeat).

**Threshold is FIXED at 0.5** (`configs/eval_cv_repeats.yaml`), not
adaptive -- unlike `07c`'s cost-sensitive default. Every repeat's
`threshold_used`/`threshold_method` is still logged (will read `0.5`/
`"fixed"` throughout) so this stays visible rather than silently assumed.

**Head depth now sweeps `linear` vs `mlp2`** for every scenario here,
same as `07`/`07b`/`07c`/`07d` — this branch originally fixed it at
`mlp2` only, treating the depth question as already settled by the
earlier branches, but that call is reopened here since the mlp2-only
results looked unsatisfying. `HEAD_DEPTHS = ["linear", "mlp2"]` is set
once, same convention `07c` already uses.

**Scenario F is now a real, trained scenario** (previously a deferred
`NotImplementedError` placeholder in `models.py`). F merges each point's
SVG+TVG graph pair into ONE HeteroData via a new `src/unified_graph.py`
(handles the `building`/`building` name collision between SVG's and
TVG's separate "building" node types by renaming to `svg_building`/
`tvg_building`) plus a new bidirectional `same_location` edge linking
SVG's `ego` node to TVG's `incident` node — present for EVERY point,
positive or negative, since both anchors always exist regardless of
label. `models.UnifiedEncoder` runs one HeteroConv stack over this
merged type space; `train.run_forward_unified` / `train_one_fold`'s
`scenario == "F"` branch and `graph_datasets.collate_pairs_unified`
handle F's one-merged-graph batch shape (every other scenario still
gets the original two-graph svg_batch/tvg_batch shape, unchanged).

**5 repeats** (`n_repeats: 5` in `configs/eval_cv_repeats.yaml`) — fewer
than `07c`'s 20, since this run's primary purpose is the scheme/city
comparison across A-G, not a high-precision variance estimate on its own.

**No formal significance testing here** — same caveat as `07b`/`07c`/
`07d`: Wilcoxon/Nadeau-Bengio are built for paired, correlated k-fold
scores; repeated stratified re-splits aren't that either (even though
every scenario shares the same rows at a given repeat, the SET of
repeats itself isn't a formal fold partition). Descriptive aggregates
(mean +/- std across the 5 repeats) only.

GPU recommended, not required (graphs are small).

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched src/ files until pushed to GitHub.
# Skip this cell once the repo itself is updated -- needs unified_graph.py
# (new), plus the edited train.py / models.py / graph_datasets.py /
# baseline_features.py (PooledDualGraphDataset, collate_pairs_unified,
# UnifiedEncoder, run_scenario_cv_repeats, _stratified_split_by_city).
from google.colab import files
import shutil

print("Upload train.py, models.py, graph_datasets.py, unified_graph.py, "
      "baseline_features.py, plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/eval_cv_repeats.yaml") as f:
    eval_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_cv_repeats.yaml") as f:
    model_cfg = yaml.safe_load(f)

# paths.yaml is per-city (see configs/paths.yaml's `per_city` map,
# generated by 00_setup_config) -- there is no single flat
# processed_dir/outputs_dir for the pooled track. Pooled outputs live
# under the `combined` section instead (same base path
# 05b_dataset_assembly_pooled.ipynb itself writes dataset_index.parquet
# to), so this branch reads paths.yaml's structure directly rather than
# assuming the flat keys 07/07b/07c/07d happen to read (those apply to a
# single active city only, per paths.yaml's own top-of-file comment
# in 00_setup_config's generation logic).
_bogor_base = Path(paths_cfg["per_city"]["bogor"]["base_dir"])
COMBINED_BASE_DIR = _bogor_base.parent / "combined"
COMBINED_PROCESSED_DIR = COMBINED_BASE_DIR / "processed"
OUTPUTS_DIR = Path(paths_cfg["combined"]["outputs_dir"]) if "combined" in paths_cfg \
    else COMBINED_BASE_DIR / "outputs"
# separate checkpoint/metrics dirs from 07/07b/07c/07d -- keeps this
# branch's results from colliding with any of the other four
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_cv_repeats"
METRICS_DIR = OUTPUTS_DIR / "metrics_cv_repeats"
for d in [CHECKPOINT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
config = {"batch_size": eval_cfg.get("batch_size") or 128,
          "epoch_cap": eval_cfg.get("epoch_cap") or 400,
          "warmup_epochs": eval_cfg.get("warmup_epochs") or 100,
          "patience": eval_cfg.get("patience") or 40,
          "lr_patience": eval_cfg.get("lr_patience") or 10,
          "lr": eval_cfg.get("lr") or 5e-3,
          "weight_decay": eval_cfg.get("weight_decay") or 1e-4,
          "fusion_dim": model_cfg.get("fusion_dim") or 128,
          # 65/15/20: val_frac=0.15, test_frac=0.20, remaining 0.65 is train
          "val_frac": eval_cfg.get("val_frac") or 0.15,
          "test_frac": eval_cfg.get("test_frac") or 0.20,
          # joint (label x city) stratification -- see
          # train._stratified_split_by_city's docstring
          "label_col": eval_cfg.get("label_col") or "label",
          "city_col": eval_cfg.get("city_col") or "city",
          "target_pos_frac": eval_cfg.get("target_pos_frac"),
          "threshold_method": eval_cfg.get("threshold_method") or "fixed",
          "threshold_fn_cost": eval_cfg.get("threshold_fn_cost") or 10.0,
          "threshold_fp_cost": eval_cfg.get("threshold_fp_cost") or 1.0,
          "num_workers": eval_cfg.get("num_workers") or 0,
          "use_amp": eval_cfg.get("use_amp", True)}
N_REPEATS = eval_cfg.get("n_repeats") or 5
print(f"Device: {device} | n_repeats: {N_REPEATS}")
print(f"Split: {1 - config['val_frac'] - config['test_frac']:.0%}/{config['val_frac']:.0%}/{config['test_frac']:.0%} "
      f"(train/val/test), stratified by ('{config['label_col']}' x '{config['city_col']}'), "
      f"target_pos_frac={config['target_pos_frac']}")
print(f"Threshold: {config['threshold_method']}"
      + (f" (fn_cost={config['threshold_fn_cost']}, fp_cost={config['threshold_fp_cost']})"
         if config['threshold_method'] == "cost_sensitive" else ""))
print(f"Warmup: {config['warmup_epochs']} epochs, patience: {config['patience']}, epoch_cap: {config['epoch_cap']}")

In [ ]:
import json
import pandas as pd
import graph_datasets as ds
import train as tr
import evaluate as ev
import models

INDEX_PATH = COMBINED_PROCESSED_DIR / "dataset_index.parquet"
index_df = pd.read_parquet(INDEX_PATH)
assert "city" in index_df.columns, (
    f"'{INDEX_PATH}' has no 'city' column -- this notebook needs 05b's "
    "pooled index, not a single city's dataset_index.parquet from 05.")
assert "uid" in index_df.columns, "expected 05b's city-prefixed 'uid' primary key column."

# PooledDualGraphDataset reads svg_dir/tvg_dir PER ROW (05b's columns) --
# NOT the single shared-dir DualGraphDataset the other 07 branches use,
# since Bogor and Warsaw's graphs live in different Drive folders.
dataset = ds.PooledDualGraphDataset(index_df)
print(f"Dataset: {len(dataset)} points (pooled, fixed-composition CV branch)")
print(index_df.groupby("city")["label"].agg(["count", "sum"]).rename(columns={"sum": "n_positive"}))

# Unified vocab sizes, read from the post-04b cache (same pattern 07f uses) --
# NOT hardcoded, since 04b's Bogor-union-Warsaw vocab is larger than either
# city's own pre-unification vocab. Either city's cache holds the same
# unified vocab post-04b (04b's last cell writes it to both).
_unified_cache_dir = _bogor_base / "interim" / "osm_cache"
with open(_unified_cache_dir / "highway_vocab.json") as f:
    HIGHWAY_VOCAB_SIZE = len(json.load(f))
with open(_unified_cache_dir / "building_type_vocab.json") as f:
    BUILDING_TYPE_VOCAB_SIZE = len(json.load(f))
print(f"Unified vocab (post-04b): highway={HIGHWAY_VOCAB_SIZE}, building_type={BUILDING_TYPE_VOCAB_SIZE}")

svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.45),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2, cat_embed_dim=2)
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.45),
                   building_type_vocab=BUILDING_TYPE_VOCAB_SIZE, highway_vocab=HIGHWAY_VOCAB_SIZE,
                   building_type_embed_dim=8, highway_embed_dim=4)


In [ ]:
# ── Train every primary scenario x head depth ─────────────────────────
# Same rationale as 07/07b/07c: one cell per scenario, split out for
# independent run/monitor/interrupt. Previously fixed at mlp2 only (see
# notebook intro) -- reopened since mlp2-only results looked unsatisfying.
PRIMARY_SCENARIOS = ["A", "B", "C", "D", "E", "F"]
HEAD_DEPTHS = ["linear", "mlp2"]
all_results = {}

### Scenario A — SVG only

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"A_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("A", depth, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario B — TVG only

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"B_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("B", depth, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario C — dual graph (concat)

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"C_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("C", depth, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario D — dual graph (late fusion)

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"D_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("D", depth, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario E — dual graph (cross-attention)

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"E_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("E", depth, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario F — unified merged graph (NEW: now a real, trained scenario)

Builds on `models.UnifiedEncoder` / `train.run_forward_unified` /
`graph_datasets.collate_pairs_unified` / `src/unified_graph.py`'s
`merge_svg_tvg` (SVG+TVG merged into one HeteroData per point, `building`
name collision resolved via `svg_building`/`tvg_building` rename, new
bidirectional `ego<->incident` `same_location` edge present for every
point regardless of label). No `svg_kwargs`/`tvg_kwargs` split needed at
the call site — `models.build_model("F", ...)` unions both internally.

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"F_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("F", depth, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Ablation B+ through F+

Same rationale as `07`/`07b`/`07c`: one cell per scenario, split out for
independent run/monitor/interrupt. F is now included in the ablation
scope (see `configs/model_cv_repeats.yaml`'s `ablation_scope`), since
it's no longer a deferred placeholder.

#### Ablation B+

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"B_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("B", depth, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

#### Ablation C+

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"C_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("C", depth, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

#### Ablation D+

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"D_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("D", depth, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

#### Ablation E+

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"E_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("E", depth, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

#### Ablation F+

In [ ]:
for depth in HEAD_DEPTHS:
    key = f"F_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_cv_repeats("F", depth, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario G — XGBoost, separate path

Tabular flattened features via `baseline_features.build_feature_table_pooled`
(reads `svg_dir`/`tvg_dir` PER ROW, same reason `PooledDualGraphDataset`
does above) and `train._stratified_split_by_city` directly (G has no
GNN encoder, so it never goes through `train_one_fold`/
`run_scenario_cv_repeats` — same standalone-path convention `07c`/`07d`
already use for G).

In [ ]:
import baseline_features
from xgboost import XGBClassifier

feat_table = baseline_features.build_feature_table_pooled(index_df, torch)
feat_table = feat_table.merge(index_df[["uid", "label", "city"]], on="uid")
feature_cols = [c for c in feat_table.columns if c not in ["uid", "label", "city"]]

g_results = []
for repeat in range(N_REPEATS):
    repeat_seed = 42 + repeat
    # SAME split scheme as A-F at this repeat index -- joint (label x
    # city) stratification, 65/15/20 -- so G is evaluated like-for-like,
    # not just on a comparable ratio.
    train_, val_, test = tr._stratified_split_by_city(
        feat_table, config["label_col"], config["city_col"],
        config["val_frac"], config["test_frac"], repeat_seed, config.get("target_pos_frac"))

    clf = XGBClassifier(n_estimators=200, max_depth=4, eval_metric="aucpr", random_state=42)
    clf.fit(train_[feature_cols], train_["label"])

    if config["threshold_method"] == "fixed":
        chosen_threshold = 0.5
        threshold_method_used = "fixed"
    else:
        val_prob = clf.predict_proba(val_[feature_cols])[:, 1]
        chosen_threshold, threshold_method_used, _ = ev.find_optimal_threshold(
            val_["label"].values, val_prob, method=config["threshold_method"],
            fn_cost=config["threshold_fn_cost"], fp_cost=config["threshold_fp_cost"])

    prob = clf.predict_proba(test[feature_cols])[:, 1]
    metrics = ev.compute_metrics(test["label"].values, prob, threshold=chosen_threshold)
    metrics["threshold_method"] = threshold_method_used
    g_results.append({"repeat": repeat, "n_train": len(train_), "n_val": len(val_), "n_test": len(test),
                       **metrics})

all_results["G"] = g_results
print(f"Scenario G: {len(g_results)} repeat-runs complete.")

## Aggregate + report every scenario

In [ ]:
agg_rows = []
for key, results in all_results.items():
    agg = ev.aggregate_fold_results(results)
    row = {"scenario": key}
    for metric, (mean, std) in agg.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(METRICS_DIR / "all_scenarios_summary_cv_repeats.csv", index=False)
display(agg_df)

## Per-city split composition check

This is a COMPOSITION check, not a per-city performance breakdown --
re-deriving each repeat's split (same `repeat_seed` training used) and
confirming each city's row count and positive rate in the test split
landed as expected under joint (label x city) stratification. It does
NOT re-score any trained model split out by city; that would need each
repeat's best-checkpoint weights evaluated separately against a
city-filtered test loader, which none of `run_scenario_cv_repeats`'s
saved artifacts support directly (only ONE best-checkpoint is kept per
scenario tag, from whichever repeat had the best val PR-AUC -- not one
per repeat). Add a dedicated per-city re-evaluation pass later if the
composition here looks fine but a city-specific performance question
remains open.

In [ ]:
def _split_composition_by_city(tag):
    """Re-derive each repeat's split (same repeat_seed used in training)
    and report each city's row count + positive rate in train/val/test --
    a composition check confirming joint (label x city) stratification
    actually landed as expected, not a re-scored performance metric."""
    rows = []
    for repeat in range(N_REPEATS):
        seed = 42 + repeat
        train_df, val_df, test_df = tr._stratified_split_by_city(
            index_df, config["label_col"], config["city_col"],
            config["val_frac"], config["test_frac"], seed, config.get("target_pos_frac"))
        for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
            for city in sorted(index_df["city"].unique()):
                city_split = split_df[split_df["city"] == city]
                rows.append({"scenario": tag, "repeat": repeat, "split": split_name, "city": city,
                             "n": len(city_split),
                             "pos_rate": city_split["label"].mean() if len(city_split) else float("nan")})
    return rows

composition_rows = []
for key in all_results:
    composition_rows.extend(_split_composition_by_city(key))

composition_df = pd.DataFrame(composition_rows)
composition_df.to_csv(METRICS_DIR / "split_composition_by_city_cv_repeats.csv", index=False)
display(composition_df[composition_df["scenario"] == "A_mlp2"].pivot_table(
    index=["split", "city"], values=["n", "pos_rate"], aggfunc="mean"))

## Threshold diagnostics

In [ ]:
threshold_rows = []
for key, results in all_results.items():
    method_counts = ev.summarize_categorical_field(results, "threshold_method")
    thresh_mean, thresh_std = ev.aggregate_fold_results(results).get("threshold_used", (float("nan"), float("nan")))
    threshold_rows.append({"scenario": key, "threshold_mean": thresh_mean, "threshold_std": thresh_std,
                            "methods_used": method_counts})

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(METRICS_DIR / "threshold_diagnostics_cv_repeats.csv", index=False)
display(threshold_df)

## Head-depth comparison

In [ ]:
# ── Head-depth comparison: descriptive only (no formal significance test
#    here -- see notebook intro for why). ──
depth_compare = []
for scenario in PRIMARY_SCENARIOS:
    lin = agg_df[agg_df["scenario"] == f"{scenario}_linear"]["pr_auc_mean"].iloc[0]
    mlp = agg_df[agg_df["scenario"] == f"{scenario}_mlp2"]["pr_auc_mean"].iloc[0]
    depth_compare.append({"scenario": scenario, "linear_pr_auc": lin, "mlp2_pr_auc": mlp})

depth_df = pd.DataFrame(depth_compare)
display(depth_df)

WINNING_DEPTH = "linear" if depth_df["linear_pr_auc"].mean() >= depth_df["mlp2_pr_auc"].mean() else "mlp2"
print(f"\nWinning head depth (by mean PR-AUC across scenarios): {WINNING_DEPTH}")
print("Descriptive only -- no Wilcoxon/Nadeau-Bengio run in this branch (see intro).")

## Epoch-level diagnostics

Per-repeat training history saved under
`CHECKPOINT_DIR/{tag}_history/repeat{N}.json`, same JSON shape as
`07b`/`07c`.

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = CHECKPOINT_DIR / "A_linear_history" / "repeat0.json"
history = json.loads(history_path.read_text())

epochs = [h["epoch"] for h in history]
best_epoch = max(range(len(history)), key=lambda i: history[i]["val_pr_auc"])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(epochs, [h["train_loss"] for h in history], label="train_loss", color="tab:blue")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train_loss", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(epochs, [h["val_pr_auc"] for h in history], label="val_pr_auc", color="tab:orange")
ax2.plot(epochs, [h["val_auroc"] for h in history], label="val_auroc", color="tab:green")
ax2.axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch ({best_epoch})")
ax2.set_ylabel("val metric")

fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
plt.title(f"repeat0 history ({history_path.name})")
plt.tight_layout()
plt.show()

In [ ]:
print("Fixed-composition CV branch complete.")
print("Every scenario A-G (including F) trained under identical 65/15/20,")
print("joint (label x city)-stratified splits at each of the 5 repeats.")
print("Compare against 07c (60/20/20, label-only stratified, single-city or")
print("unstratified-by-city pooling) to see whether city-aware stratification")
print("and/or the merged-graph F scenario change the ranking across A-G.")